In [3]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression 
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

In [4]:
df = pd.read_csv("../data/raw/wait_times.csv")
df.head


<bound method NDFrame.head of                     port_name             category lane_type          status  \
0      Bridge of the Americas  Commercial Vehicles   General          closed   
1      Bridge of the Americas  Commercial Vehicles      Fast          closed   
2      Bridge of the Americas   Passenger Vehicles   General        reported   
3      Bridge of the Americas   Passenger Vehicles    Sentri  not applicable   
4      Bridge of the Americas   Passenger Vehicles     Ready        reported   
...                       ...                  ...       ...             ...   
19651                 Stanton   Passenger Vehicles   General          closed   
19652                 Stanton   Passenger Vehicles    Sentri        reported   
19653                 Stanton   Passenger Vehicles     Ready  not applicable   
19654                 Stanton           Pedestrian   General  not applicable   
19655                 Stanton           Pedestrian     Ready  not applicable   

       wa

In [5]:
pd.to_numeric(df['wait_minutes'], errors='coerce').notnull()

0        False
1        False
2         True
3        False
4         True
         ...  
19651    False
19652     True
19653    False
19654    False
19655    False
Name: wait_minutes, Length: 19656, dtype: bool

In [6]:
df_clean = df[pd.to_numeric(df['wait_minutes'], errors='coerce').notnull()]
df['status'].value_counts()

status
reported          8860
not applicable    7033
closed            3763
Name: count, dtype: int64

Reported: 6782

NA: 5391

Closed: 2891

In [7]:
df_clean['timestamp'] = pd.to_datetime(df_clean['timestamp'])
df_clean.head(1000)

,port_name,category,lane_type,status,wait_minutes,timestamp
2,Bridge of the Americas,Passenger Vehicles,General,reported,50.0,2026-08-09 15:26:14.176934
4,Bridge of the Americas,Passenger Vehicles,Ready,reported,45.0,2026-08-09 15:26:14.176934
5,Bridge of the Americas,Pedestrian,General,reported,1.0,2026-08-09 15:26:14.176934
6,Bridge of the Americas,Pedestrian,Ready,reported,1.0,2026-08-09 15:26:14.176934
9,Paso Del Norte,Passenger Vehicles,General,reported,54.0,2026-08-09 15:26:14.176934
...,...,...,...,...,...,...
1970,Paso Del Norte,Passenger Vehicles,Sentri,reported,1.0,2026-08-14 19:40:48.703565
1971,Paso Del Norte,Passenger Vehicles,Ready,reported,30.0,2026-08-14 19:40:48.703565
1972,Paso Del Norte,Pedestrian,General,reported,8.0,2026-08-14 19:40:48.703565
1973,Paso Del Norte,Pedestrian,Ready,reported,5.0,2026-08-14 19:40:48.703565


In [8]:
df_clean['hour'] = df_clean['timestamp'].dt.hour
df_clean['day_of_week'] = df_clean['timestamp'].dt.day_of_week

df_clean.head(50)

,port_name,category,lane_type,status,wait_minutes,timestamp,hour,day_of_week
2,Bridge of the Americas,Passenger Vehicles,General,reported,50.0,2026-08-09 15:26:14.176934,15,6
4,Bridge of the Americas,Passenger Vehicles,Ready,reported,45.0,2026-08-09 15:26:14.176934,15,6
5,Bridge of the Americas,Pedestrian,General,reported,1.0,2026-08-09 15:26:14.176934,15,6
6,Bridge of the Americas,Pedestrian,Ready,reported,1.0,2026-08-09 15:26:14.176934,15,6
9,Paso Del Norte,Passenger Vehicles,General,reported,54.0,2026-08-09 15:26:14.176934,15,6
10,Paso Del Norte,Passenger Vehicles,Sentri,reported,5.0,2026-08-09 15:26:14.176934,15,6
11,Paso Del Norte,Passenger Vehicles,Ready,reported,50.0,2026-08-09 15:26:14.176934,15,6
12,Paso Del Norte,Pedestrian,General,reported,8.0,2026-08-09 15:26:14.176934,15,6
13,Paso Del Norte,Pedestrian,Ready,reported,5.0,2026-08-09 15:26:14.176934,15,6
17,Ysleta,Passenger Vehicles,Sentri,reported,1.0,2026-08-09 15:26:14.176934,15,6


In [9]:
df['timestamp'].dtype
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['timestamp'].head(10)

0   2026-08-09 15:26:14.176934
1   2026-08-09 15:26:14.176934
2   2026-08-09 15:26:14.176934
3   2026-08-09 15:26:14.176934
4   2026-08-09 15:26:14.176934
5   2026-08-09 15:26:14.176934
6   2026-08-09 15:26:14.176934
7   2026-08-09 15:26:14.176934
8   2026-08-09 15:26:14.176934
9   2026-08-09 15:26:14.176934
Name: timestamp, dtype: datetime64[us]

Implements cyclical feature encoding for both hour and day_of_week columns

In [10]:
df_clean.index
df_clean['hour_sin'] = np.sin(2 * np.pi * df_clean['hour'] / 24)
df_clean['hour_cos'] = np.cos(2 * np.pi * df_clean['hour'] / 24)

df_clean['day_of_week_sin'] = np.sin(2 * np.pi * df_clean['day_of_week'] / 7)
df_clean['day_of_week_cos'] = np.cos(2 * np.pi * df_clean['day_of_week'] / 7)

df_clean



,port_name,category,lane_type,status,wait_minutes,timestamp,hour,day_of_week,hour_sin,hour_cos,day_of_week_sin,day_of_week_cos
2,Bridge of the Americas,Passenger Vehicles,General,reported,50.0,2026-08-09 15:26:14.176934,15,6,-0.707107,-0.707107,-0.781831,0.62349
4,Bridge of the Americas,Passenger Vehicles,Ready,reported,45.0,2026-08-09 15:26:14.176934,15,6,-0.707107,-0.707107,-0.781831,0.62349
5,Bridge of the Americas,Pedestrian,General,reported,1.0,2026-08-09 15:26:14.176934,15,6,-0.707107,-0.707107,-0.781831,0.62349
6,Bridge of the Americas,Pedestrian,Ready,reported,1.0,2026-08-09 15:26:14.176934,15,6,-0.707107,-0.707107,-0.781831,0.62349
9,Paso Del Norte,Passenger Vehicles,General,reported,54.0,2026-08-09 15:26:14.176934,15,6,-0.707107,-0.707107,-0.781831,0.62349
...,...,...,...,...,...,...,...,...,...,...,...,...
19643,Ysleta,Commercial Vehicles,Fast,reported,25.0,2026-11-02 23:55:31.684568,23,0,-0.258819,0.965926,0.000000,1.00000
19645,Ysleta,Passenger Vehicles,Sentri,reported,1.0,2026-11-02 23:55:31.684568,23,0,-0.258819,0.965926,0.000000,1.00000
19646,Ysleta,Passenger Vehicles,Ready,reported,60.0,2026-11-02 23:55:31.684568,23,0,-0.258819,0.965926,0.000000,1.00000
19647,Ysleta,Pedestrian,General,reported,5.0,2026-11-02 23:55:31.684568,23,0,-0.258819,0.965926,0.000000,1.00000


Encodes Boolean values for port_name, category, and lane_type

In [11]:
df_encoded = pd.get_dummies(df_clean, columns=['port_name', 'category', 'lane_type']).drop(columns=[
    'hour', 'timestamp', 'day_of_week', 'status'])

df_encoded.columns.tolist()


['wait_minutes',
 'hour_sin',
 'hour_cos',
 'day_of_week_sin',
 'day_of_week_cos',
 'port_name_Bridge of the Americas',
 'port_name_Paso Del Norte',
 'port_name_Stanton',
 'port_name_Ysleta',
 'category_Commercial Vehicles',
 'category_Passenger Vehicles',
 'category_Pedestrian',
 'lane_type_Fast',
 'lane_type_General',
 'lane_type_Ready',
 'lane_type_Sentri']

Use Boolean Masking on df_encoded to split into a train and test set

In [12]:
cutoff = df_clean['timestamp'].quantile(0.8)
mask = df_clean['timestamp'] <= cutoff

df_train = df_encoded[mask]
df_test = df_encoded[~mask]

df_train_unencoded = df_clean[mask]
df_test_unencoded = df_clean[~mask]

len(df_train) / len(df_encoded)

0.8003386004514673

In [13]:
X_train = df_train.drop(columns=['wait_minutes'])
y_train = df_train['wait_minutes']

X_test = df_test.drop(columns=['wait_minutes'])
y_test = df_test['wait_minutes']
print(X_train.shape[1])
print(y_train.shape[0])

15
7091


In [14]:
lookup = df_train_unencoded.groupby(['port_name', 'hour'])['wait_minutes'].mean().reset_index()
lookup

,port_name,hour,wait_minutes
0,Bridge of the Americas,0,34.155556
1,Bridge of the Americas,1,33.093750
2,Bridge of the Americas,2,28.779661
3,Bridge of the Americas,3,31.508475
4,Bridge of the Americas,4,27.162791
...,...,...,...
86,Ysleta,19,23.269231
87,Ysleta,20,22.699187
88,Ysleta,21,22.682432
89,Ysleta,22,21.084034


In [15]:
df_test_unencoded.groupby(['port_name', 'hour'])['wait_minutes'].mean()

port_name               hour
Bridge of the Americas  0       44.333333
                        1       45.048780
                        2       33.500000
                        3       55.333333
                        4       23.875000
                                  ...    
Ysleta                  19      18.722222
                        20      19.923077
                        21      17.951220
                        22      23.333333
                        23      17.173913
Name: wait_minutes, Length: 90, dtype: float64

In [16]:
df_unencoded_merge = pd.merge(df_test_unencoded, lookup, on=['port_name', 'hour'], how='left', suffixes=['_actual', '_predicted'])
print(df_unencoded_merge)



merged_clean = df_unencoded_merge.dropna(subset=['wait_minutes_predicted'])

                   port_name             category lane_type    status  \
0     Bridge of the Americas   Passenger Vehicles   General  reported   
1     Bridge of the Americas   Passenger Vehicles     Ready  reported   
2     Bridge of the Americas           Pedestrian   General  reported   
3             Paso Del Norte   Passenger Vehicles   General  reported   
4             Paso Del Norte   Passenger Vehicles    Sentri  reported   
...                      ...                  ...       ...       ...   
1764                  Ysleta  Commercial Vehicles      Fast  reported   
1765                  Ysleta   Passenger Vehicles    Sentri  reported   
1766                  Ysleta   Passenger Vehicles     Ready  reported   
1767                  Ysleta           Pedestrian   General  reported   
1768                 Stanton   Passenger Vehicles    Sentri  reported   

      wait_minutes_actual                  timestamp  hour  day_of_week  \
0                    40.0 2026-10-04 13:17:24.28

Use an MAE to establish a baseline for the other models to beat.

In [17]:
baseline_mae = mean_absolute_error(merged_clean['wait_minutes_actual'], merged_clean['wait_minutes_predicted'])
baseline_mae

20.001918844998947

Super Duper Dumb Baseline

In [18]:
median_guess = y_train.median()
median_guess_array = np.full(y_test.size, median_guess)
dumb_dumb_baseline = mean_absolute_error(y_test, median_guess_array)
print(dumb_dumb_baseline)

19.87111362351611


In [19]:
lookup_upgraded = df_train_unencoded.groupby(['port_name', 'hour', 'category', 'lane_type'])['wait_minutes'].mean().reset_index()
lookup_upgraded

,port_name,hour,category,lane_type,wait_minutes
0,Bridge of the Americas,0,Passenger Vehicles,General,57.925926
1,Bridge of the Americas,0,Passenger Vehicles,Ready,51.962963
2,Bridge of the Americas,0,Pedestrian,General,3.148148
3,Bridge of the Americas,0,Pedestrian,Ready,2.444444
4,Bridge of the Americas,1,Passenger Vehicles,General,52.600000
...,...,...,...,...,...
362,Ysleta,23,Commercial Vehicles,General,31.080000
363,Ysleta,23,Passenger Vehicles,General,50.000000
364,Ysleta,23,Passenger Vehicles,Ready,50.675676
365,Ysleta,23,Passenger Vehicles,Sentri,5.105263


In [20]:
df_unencoded_merge_upgraded = pd.merge(df_test_unencoded, lookup_upgraded, on=['port_name', 'hour', 'category', 'lane_type'], how='left', suffixes=['_actual', '_predicted'])
print(df_unencoded_merge_upgraded['wait_minutes_predicted'])



merged_clean_upgraded = df_unencoded_merge_upgraded.dropna(subset=['wait_minutes_predicted'])


0       39.833333
1       35.416667
2       10.333333
3       46.625000
4        3.250000
          ...    
1764    21.200000
1765     5.105263
1766    50.675676
1767     3.421053
1768     3.473684
Name: wait_minutes_predicted, Length: 1769, dtype: float64


In [ ]:
df_unencoded_merge_upgraded['wait_minutes_predicted'].isnull().sum()

np.int64(2)

Upgraded Baseline MAE with category and lane_type columns added

In [ ]:
upgraded_baseline_mae = mean_absolute_error(merged_clean_upgraded['wait_minutes_actual'], merged_clean_upgraded['wait_minutes_predicted'])
print(upgraded_baseline_mae)

9.81301013376443


Linear Regression Model

In [ ]:
linear_regression_model = LinearRegression()
linear_regression_model.fit(X_train, y_train)
linear_model_predictions = linear_regression_model.predict(X_test)


In [ ]:
linear_regression_mae = mean_absolute_error(y_test, linear_model_predictions)
print(linear_regression_mae)

10.738541166632121


Random Forest Regressor Model

In [ ]:
rfr_model = RandomForestRegressor(n_estimators=100, random_state=42)
rfr_model.fit(X_train, y_train)
rfr_model_predictions = rfr_model.predict(X_test)

print(mean_absolute_error(y_train, rfr_model.predict(X_train)))

5.1350730198470815


In [ ]:
rfr_mae = mean_absolute_error(y_test, rfr_model_predictions)
print(rfr_mae)

8.072578278772827
